# rpart 教程（0/3）：总览与环境

> **本文件在系列中的位置**：这是 rpart 系列教程的**第 0 部分**，负责交代「包是什么、能做什么、函数地图、术语、控制参数、1-SE 剪枝的就地定义、绘图环境」。读完后请进入 `01_建树与剪枝机制.ipynb`。
>
> **系列结构**：
> - **本文件（0）**：预备 + 附录 A（predict 速查）+ 附录 B（常见坑）
> - `01_建树与剪枝机制.ipynb`：不纯度、cp、cp 表、1-SE、三棵树对比 + 附录 C（1-SE 深探）
> - `02_按method展开.ipynb`：class / anova / poisson / 生存 + 附录 D（其余函数与数据集）
> - `03_横切专题.ipynb`：代价敏感、surrogate、重要性、复杂度 + 附录 E（Breiman 排序定理证明）
>
> **自包含约定**：各文件代码块沿用「就地定义辅助函数」的写法，可独立运行；跨文件的 cp 表读法与 1-SE 规则在第 1 部分讲透，后续部分只引用不重复。

---

## 本文件导读

本文件是「预备篇」，不涉及具体建模。内容顺序：

1. **0.1 包是什么**：CART 血统、能力边界、性能画像；
2. **0.2 函数全览**：函数地图，按建树/控制/预测/剪枝/诊断/可视化/数据集分类；
3. **0.3 设计特点与基本用法**：统一接口、method 自动选择规则、最小示例；
4. **0.4 术语表**：cp、xerror、xstd、1-SE、surrogate、improve 等；
5. **0.5 控制参数总表**：`rpart.control` 全部参数与默认值；
6. **0.6 1-SE 剪枝与 `prune_1se()`**：为什么用 1-SE、函数实现、生活化比喻；
7. **0.7 图表环境设置**：关闭警告、调整打印宽度与画幅；
8. **附录 A**：`predict` 输出速查表；
9. **附录 B**：常见坑清单（17 条）。

读完本文件，你应能：说清 rpart 与 ID3/C4.5 的区别；查表找到任意函数；写出最小建树/预测代码；理解 1-SE 规则的动机与实现。

# rpart：递归分割与回归树

本 notebook 系统介绍 R 语言 `rpart` 包。整体结构遵循「**包介绍 → 机制 → 方法 → 专题 → 附录**」的主线：

- **第 0 节 预备**：rpart 包是什么、能做什么、**函数全览**、设计特点、安装与基本用法；术语表、控制参数总表、统一环境设置。
- **第 1 节 建树与剪枝机制**：分支原理（不纯度）、cp、xerror、xstd、1-SE 规则、`prune_1se`、cp 表逐列解读——所有 method 共用的核心，集中讲透，后续不再重复。
- **第 2 节 按 method 展开**：`class` → `anova` → `poisson` → 生存分析，每个 method 只讲它特有的部分。
- **第 3 节 横切专题**：代价敏感三法、缺失值与 surrogate、变量重要性、时间复杂度——与 method 正交，单独成节。
- **附录**：predict 输出速查、常见坑、**1-SE 规则的深入探讨**、**rpart 其余函数与数据集速览（补全包内全部对象）**、**Breiman 类别排序定理的详细证明**。


**依赖说明**：本教程只依赖 CRAN 上仍在维护的包（`rpart`、`partykit`、`rpart.plot`、`skimr`、`AppliedPredictiveModeling`、`plotly`、`vip`、`ggplot2`、`survival`、`dplyr`）。

参考文献：
1. Random Forests with R，Robin Genuer，Jean-Michel Poggi，2020
2. rpart 官方文档 `?rpart`、`?rpart.control`、`?predict.rpart`

算法参见（综合参考，单篇不一定详尽正确）：
- https://zhuanlan.zhihu.com/p/85731206
- https://zhuanlan.zhihu.com/p/139519852
- https://zhuanlan.zhihu.com/p/82054400

# 第 0 节 预备

## 0.1 rpart 包是什么

`rpart`（**R**ecursive **PART**itioning）是 R 中实现 **CART（Classification and Regression Trees）** 算法的主力包，由 Terry Therneau 等人维护，是 R 基础发行版推荐包之一（`recommended`，随 R 一起安装）。

**它解决什么问题：** 给定一张「自变量 + 因变量」的数据表，用一系列**轴平行**的二元分割（如 `x1 < 3.2`、`Type ∈ {F, I}`）把样本递归地切成若干组，使每组内部因变量尽量同质；最终得到一棵树，可用于**预测**和**解释**。因变量是类别 → 分类树；连续 → 回归树；计数 → 泊松树；生存时间 → 生存树。

### 0.1.1 算法血统：它依据什么

`rpart` 实现的是 **CART**（Breiman, Friedman, Olshen, Stone, 1984），而非 Quinlan 系的 ID3/C4.5/C5.0。这一血统决定了它的四个标志性设计：

1. **严格二叉树**：每次只切两叉（哪怕自变量有 K 个水平，也只在水平集合上做二分）；
2. **Gini 指数 / 方差作不纯度**：分类用 Gini，回归用 SSE/方差（而非信息增益/增益率）；
3. **代价复杂度剪枝（cp）**：先充分生长，再按 cp 自底向上剪枝，用交叉验证误差 `xerror` 与 `xstd` 选子树；
4. **代理分割（surrogate）处理缺失**：主分割变量缺失时，用与之一致性最高的其他变量顶替。

本教程第 1 节讲的 cp、1-SE 规则、第 3.2 节的 surrogate，全部源于 CART 这套设计。

### 0.1.2 能力边界：它擅长什么、不擅长什么

**它擅长：**
- 中小规模**表格数据**上的建模与规则提取；
- 需要**可解释规则**的场景（树可读、可画、可讲故事）；
- **缺失值多**或**错判代价不对称**的业务问题（内置 surrogate 与三种代价敏感手段，见 3.1、3.2）；
- 作为**基线模型**或**特征筛选**工具，为后续集成模型探路。

**它不擅长：**
- 追求极致预测精度——单棵树方差大，通常不如随机森林/GBDT（这是单树模型的通病，不是 rpart 独有的缺陷）；
- **斜向决策边界**——轴平行分割要用很多步阶梯去逼近一条斜线，树会变深、变碎；
- **超高基数无序类别变量**——二分组合搜索慢（见 1.2 节与附录 E 的边界讨论）；
- **超大规模数据**——单机内存受限，需换用分布式/外存实现。

**性能画像（定性，实测见第 3.4 节）：**

| 维度 | rpart 的表现 | 依据 |
|---|---|---|
| 决策边界 | 轴平行，斜边界需多步逼近 | CART 二叉树 |
| 过拟合控制 | 强，cp + 1-SE 内置 | 代价复杂度剪枝 |
| 缺失值 | 原生支持，surrogate 顶替 | CART surrogate |
| 代价敏感 | 原生支持，先验/代价/权重三法等价 | CART 代价敏感 |
| 精度上限 | 单树，低于集成 | 方差大 |
| 训练复杂度 | 近线性于 n，对 p 敏感 | 每层排序 O(p·n log n) |
| 可解释性 | 强，规则可读 | 单树结构 |

> 一句话定位：**rpart 是「可解释、鲁棒、工程友好」的单树基线首选；要冲精度再上集成，要吃透树机制就从它入手。**

## 0.2 rpart 包的函数全览（函数地图）

`rpart` 包本身的函数并不多，但五脏俱全，覆盖了从建树到预测的全流程。为了方便查阅，将其核心函数按功能分类如下：

| 功能分类 | 核心函数 | 说明 |
|---|---|---|
| **建树** | `rpart()` | 最核心的函数，负责训练决策树。通过 `method` 参数自动适配分类、回归、生存分析等。 |
| **控制参数** | `rpart.control()` | 用于设定树的生长规则（如 `cp`、`maxdepth`、`minsplit`、`xval` 等），通常在 `rpart()` 内通过 `control = rpart.control(...)` 调用。 |
| **预测** | `predict.rpart()` | 对训练好的树模型进行预测。支持输出类别、概率、连续值等（依据 `type` 参数）。 |
| **剪枝** | `prune()` / `prune.rpart()` | 根据 `cp` 值对一棵充分生长的树进行剪枝，提取子树。（注：本教程为了方便，大量使用了自定义的 `prune_1se()` 自动执行 1-SE 剪枝，见 0.6）。 |
| **查看与摘要** | `print.rpart()`<br>`summary.rpart()` | `print` 快速输出树的分支结构；`summary` 输出详细的模型摘要，包括 cp 表、变量重要性、各节点的分裂优度等。 |
| **可视化** | `plot.rpart()`<br>`text.rpart()`<br>`labels.rpart()`<br>`post()` / `post.rpart()` | `plot` 绘制基础树结构图，`text` 在节点旁添加文字，`labels` 生成分割标签，`post` 输出 PostScript 树图。*(注：业界更推荐 `rpart.plot::rpart.plot()` 或 `partykit::plot(as.party())`)*。 |
| **评估与诊断** | `printcp()`<br>`plotcp()`<br>`rsq.rpart()`<br>`meanvar()` / `meanvar.rpart()`<br>`residuals.rpart()`<br>`xpred.rpart()` | `printcp` 打印 cp 表；`plotcp` 画 xerror 随 cp 曲线（横轴为相邻 cp 几何平均）；`rsq.rpart` 画各分裂的近似 R²；`meanvar` 画节点均值-方差图；`residuals` 提取残差；`xpred` 返回交叉验证预测值。 |
| **交互/辅助** | `path.rpart()`<br>`snip.rpart()`<br>`na.rpart()`<br>`rpart.exp()`<br>`rpart.object` | `path` 提取从根节点到特定节点的路径规则；`snip` 交互式手动剪枝；`na.rpart` 处理缺失值；`rpart.exp` 为指数（生存）拟合初始化；`rpart.object` 是树对象的结构定义。 |
| **数据集** | `kyphosis`<br>`solder` / `solder.balance`<br>`stagec`<br>`car.test.frame`<br>`car90`<br>`cu.summary` | rpart 包自带的经典示例数据集（汽车数据一组三个：`car.test.frame`、`car90`、`cu.summary`）。 |

> **提示**：`rpart` 的很多核心功能（如 cp 表、剪枝逻辑）在其他高级树模型包（如 `randomForest`、`xgboost`）中都有类似的影子，掌握 `rpart` 的函数地图，是入门机器学习树模型的最佳起点。本教程正文重点讲主干函数，**其余函数与数据集的速览见附录 D**。

## 0.3 设计特点与基本用法



1. **分类与回归统一接口**：同一函数 `rpart()`，按因变量类型自动选 `method`；自变量可连续可离散，且不要求相互独立。
2. **剪枝与代价敏感内置于接口**：剪枝通过 `cp` 与 `control` 控制（能力见 0.1.2、机制见第 1 节）；代价敏感通过 `parms`（先验/代价）或 `weights`（权重）实现，三法等价（用法见 3.1）。
3. **缺失值接口**：建模时用 surrogate splits 搜索替代分割，预测时按代理逐级启用；`control` 的 `maxsurrogate`/`usesurrogate` 控制搜索与启用策略（见 3.2）。
4. **变量重要性**：自带 `variable.importance` 字段，也可配合 permutation 法（见 3.3）。
5. **输出规范**：`print`/`summary`/`plot`/`predict` 方法齐全，cp 表是理解模型复杂度的核心输出。

**`method` 的自动选择规则（重要）：** `rpart()` 会依据因变量类型自动推断 `method`：

| 因变量类型 | 自动选用的 `method` |
|---|---|
| 因子（factor） | `class`（分类树） |
| 连续数值 | `anova`（回归树） |
| `Surv()` 对象 | `exp`（生存树） |

> **陷阱**：**计数因变量（如 0,1,2,… 的次数）会被当作连续数值而自动走 `anova`**，并不会自动选 `poisson`。若确实要用泊松树，必须**显式指定** `method = "poisson"`（见 2.3）。同理，二分类若因变量是数值 0/1，也会走 `anova`，需先转为因子。

**安装与加载：**

```r
install.packages("rpart")      # 通常已随 R 安装
library(rpart)
```

**最小用法示例：**

```r
fit <- rpart(y ~ ., data = df)              # 建树（默认 cp = 0.01）
print(fit)                                  # 打印树结构
print(fit$cptable)                          # cp 表（核心！见 1.3）
plot(fit); text(fit)                        # 基础绘图
predict(fit, newdata = df, type = "class")  # 预测
```

> 本 notebook 用到的辅助包：`partykit`（`as.party()` 画更清晰的树）、`rpart.plot`/`rattle`（`fancyRpartPlot`）、`skimr`（数据勘察）、`AppliedPredictiveModeling`（abalone 数据）、`plotly`、`vip`、`ggplot2`、`survival`、`dplyr`。

## 0.4 术语表

| 术语 | 英文 | 含义 |
|---|---|---|
| 复杂度参数 | cp (complexity parameter) | 控制树生长/剪枝的阈值。cp 越小树越大；cp=0 表示充分生长；cp=Inf 表示只有根节点 |
| 相对误差 | rel error | 训练集误差（以根节点误差为基准的相对值） |
| 交叉验证误差 | xerror | k 折交叉验证误差的均值（默认 10 折） |
| 交叉验证标准差 | xstd | xerror 的标准差，反映其随机性 |
| 1-SE 规则 | 1-SE rule | 在满足 `xerror ≤ min(xerror) + xstd` 的 cp 中取最大者 |
| 分割优度 | improve | 一次分割带来的 impurity（或误差）下降量 |
| 代理分割 | surrogate split | 当主分割变量缺失时，用与主分割一致性最高的其他变量代替 |
| 一致性 | agree / adj | 代理分割与原分割的一致比例 / 调整后一致比例 |
| 不纯度 | impurity | 分类中用 Gini 指数；回归中用方差（SSE） |
| 代价复杂度 | cost-complexity | $R_\alpha(T) = R(T) + \alpha |T|$，在误差与树的规模之间做权衡；rpart 中的 cp 即 α |

## 0.5 `rpart.control` 控制参数总表

| 参数 | 默认值 | 作用 |
|---|---|---|
| `cp` | 0.01 | 复杂度参数；分割必须使误差下降至少 cp 倍才保留 |
| `minsplit` | 20 | 节点至少含多少个样本才尝试分割 |
| `minbucket` | `minsplit/3` | 叶节点至少含多少个样本 |
| `maxdepth` | 30 | 树的最大深度 |
| `maxcompete` | 4 | 每个节点保留多少个竞争分割（用于输出） |
| `maxsurrogate` | 5 | 每个节点最多搜索多少个代理分割（0 表示不搜索，可加速） |
| `usesurrogate` | 2 | 0=只用主分割，缺失则不分；1=按序用代理，全缺失则不分；2=全缺失则按多数类返回 |
| `xval` | 10 | 交叉验证折数（用于计算 xerror/xstd） |
| `surrogatestyle` | 0 | 代理分割的选择风格（0=按 agree 排序，1=按 adj 排序） |

> 提示 1：`xerror` 与 `xstd` 具有随机性（源于数据随机分割）。复现时请在 `rpart()` 前 `set.seed()`。
>
> 提示 2：分类树还可通过 `rpart()` 的 `parms` 参数控制不纯度与先验：`parms = list(split = "gini")` 或 `split = "information"` 选择 Gini 指数还是信息增益作为分裂准则；`parms = list(prior = ...)`、`parms = list(loss = ...)` 用于代价敏感（见 3.1）。`parms` 不属于 `rpart.control`，而是 `rpart()` 的直接参数。

## 0.6 关于 1-SE 剪枝：就地定义的 `prune_1se()`

本 notebook 多次执行 **1-SE 剪枝**。为免在每个代码块里重复写剪枝逻辑，我们先在 0.6 给出 `prune_1se()` 的定义与原理，后续代码块内**就地定义**同名函数（保持各块自包含）。

### 0.6.1 1-SE 规则讲清楚

**1-SE 规则要解决什么问题？** 只按 `xerror` 最小值选 cp，往往会选到一棵非常复杂的树：`xerror` 的最小值可能只是交叉验证随机切分带来的「运气好」，换一组切分就未必仍最小。1-SE 规则的做法是：**在「误差与最优值无显著差异」的候选树中，选最简单的那棵。** 这里的「无显著差异」用 1 个标准误来界定。

**具体步骤：**

1. 在 cp 表中找到 `xerror` 最小的那一行，记下 `xerror_min` 与其对应的 `xstd`；
2. 设定阈值 `thr = xerror_min + 1 × xstd`；
3. 在所有满足 `xerror ≤ thr` 的行中，取**最大**的 `cp`（cp 越大树越简单）；
4. 用这个 cp 调用 `rpart::prune()`，得到剪枝后的子树。

**为什么是 1 倍标准差？** `xerror` 是 10 折交叉验证的均值，`xstd` 是其标准差，衡量了「换一组随机切分，xerror 会波动多大」。若一棵树的 xerror 落在 `xerror_min ± 1×xstd` 内，它与最优树的差异在统计上不足以判定「谁真的更好」；此时按奥卡姆剃刀选更简单的树，泛化往往更稳。0 倍会过拟合，2 倍又可能欠拟合，1 倍是常用折中。

**一个生活化比喻：** 两个长跑选手，A 平均 10.0 秒但成绩忽好忽坏（标准差 0.5），B 平均 10.2 秒但非常稳定（标准差 0.1）。只看均值会选 A；但 A 的 10.0 可能只是运气。把门槛设为 `10.0 + 0.5 = 10.5` 秒，B 的 10.2 秒也达标；若 B 还「更省事」（对应树更简单、cp 更大），就选 B。

### 0.6.2 `prune_1se()` 的实现

```r
prune_1se <- function(object, se = 1) {
  cptab <- object$cptable
  xerror <- cptab[, "xerror"]
  xstd   <- cptab[, "xstd"]
  cp     <- cptab[, "CP"]
  min_idx <- which.min(xerror)
  threshold <- xerror[min_idx] + se * xstd[min_idx]
  cp_best <- max(cp[xerror <= threshold])
  rpart::prune(object, cp = cp_best)
}
```

- 返回**剪枝后的树对象**；
- `se = 1`：标准 1-SE 规则；`se = 0` 则退化为「取 xerror 最小的 cp」。

等价于手工：在 cp 表中找 `xerror ≤ min(xerror) + se*xstd` 的最大 cp，再 `prune(tree, cp = 该值)`。

## 0.7 图表环境设置

**说明**：为避免后续代码块产生大量不影响结果的警告（如画图参数的小问题），这里关闭警告提示；同时调整控制台打印宽度和 Jupyter 的默认画幅比例（16:9），使得后续画出的决策树图和曲线图更加清晰宽大。

In [ ]:
options(warn = -1)                                   # 忽略一切警告（教学演示用）
options("width" = 140)                               # 充分利用打印宽度
options(repr.plot.width = 16, repr.plot.height = 9)  # 设置画幅比例 16:9

---

# 附录 A. predict 输出速查

| method | `type` | 返回 |
|---|---|---|
| `class` | `"class"`（默认） | 类别因子 |
| `class` | `"prob"` | 概率矩阵（列=levels） |
| `anova` | `"vector"`（默认） | 预测均值向量 |
| `anova` | `"matrix"` | nsplit × n 矩阵 |
| `poisson` | `"vector"`（默认） | 预测计数均值 |
| `poisson` | `"matrix"` | nsplit × n 矩阵 |
| `exp`（生存） | `"vector"`（默认） | 预测生存时间/风险 |
| `exp` | `"matrix"` | nsplit × n 矩阵 |

# 附录 B. 常见坑

1. **xerror/xstd 有随机性**：复现请在 `rpart()` 前 `set.seed()`。
2. **计数因变量不会自动走 poisson**：会被当作连续而走 `anova`，必须显式 `method = "poisson"`；二分类若因变量是 0/1 数值也会走 `anova`，需先转因子。
3. **`cp = 0` 在代价敏感场景可能不稳定**：用 `cp = 1e-5`；第 1 节普通场景的 cp=0 不适用于代价敏感。
4. **`plotcp` 横轴是相邻 cp 的几何平均**，不是 cp 本身；且默认绘制的水平虚线即为 1-SE 阈值。
5. **1-SE 剪枝由就地定义的 `prune_1se()` 实现**；其逻辑见 0.6，各代码块自包含该函数。
6. **缺失值手算 improve**：父节点基数用排除缺失后的样本数。
7. **自带 variable.importance 可能高估无关变量**：结合 permutation importance。
8. **`usesurrogate` 默认 2**：全缺失按多数类返回，误差 = 多数类比例。
9. **分类树默认 Gini**：可用 `parms = list(split = "information")` 改用信息增益；`parms` 是 `rpart()` 的直接参数，不属于 `rpart.control`。
10. **`post()` 已过时**：生成 PostScript 图，现代环境建议用 `rpart.plot`/`partykit` 替代。
11. **示例代码块相互独立**：每个示例块自带 `library()`、数据准备与辅助函数，可单独运行，无需提前加载任何环境或变量。
12. **Breiman 排序定理只在二分类 + Gini 或回归 + MSE 成立**；多分类或信息增益下，排序法失效，rpart 改用其它搜索。
13. **每个建模示例前先勘察数据**：统一用 `skimr::skim_tee()`；`skim_tee()` 已覆盖维度、类型、缺失、数值分布、因子水平，**勘察块不再重复 `dim()`、`str()`、`colSums(is.na())`**，只补 skim 不覆盖的派生分析。
14. **真实业务数据集建模前先交代背景**：abalone（鲍鱼年龄）、solder.balance（焊接缺陷）、stagec（前列腺癌生存）、iris（鸢尾品种）均先说明研究问题、变量现实含义与建模目标；纯模拟数据只说明生成机制。
15. **cp 的本质与计算**：cp 是代价复杂度 $R_\alpha(T)=R(T)+\alpha|T|$ 中的 α；由弱链接剪枝 $g(t)=\frac{R(t)-R(T_t)}{|T_t|-1}$ 逐步算出，cp 从 0 连续增大，每增大到某节点的 $g(t)$ 时触发一次剪枝，记录这些临界值即 cp 表各行的 CP（见第 1 部分 1.4 节）。
16. **关注类别平衡性并用代码输出**：在讲解分类模型时，平衡与不平衡是刻意设计的对比。例如第 2 部分 2.1 节刻意将阈值定为 >9 保持平衡，而第 3 部分 3.1 节刻意将阈值改为 >12 制造非平衡，以演示代价敏感的补救效果。所有勘察块必须包含 `table()` 与 `prop.table()` 直接输出频数和比例。
17. **平衡与非平衡场景下 rpart 的关注点不同**：在平衡数据下（如第 1 部分 1.1、第 2 部分 2.1 节），rpart 的默认行为完全无偏，关注点纯粹在于**如何高效、纯净地切分特征空间**（Gini、improve）；而在非平衡数据下（如第 3 部分 3.1 节），rpart 会惰性地偏向多数类，必须通过先验、代价矩阵或权重（`parms` 或 `weights`）**强行纠正偏差**，关注点转向了代价敏感与纠偏机制。

---

## 本文件小结与下一步

本文件已完成预备篇的全部内容：包定位（CART 血统、能力边界）、函数地图、method 自动选择规则、术语表、控制参数总表、1-SE 剪枝与 `prune_1se()`、绘图环境，以及附录 A（predict 速查）与附录 B（常见坑）。

**下一步**：进入 `01_建树与剪枝机制.ipynb`，用二维平面上的分类/回归例子，把「不纯度 → improve → cp → cp 表 → 1-SE → 三棵树对比」这条主线一次讲透。这些机制是所有 method 共用的核心，后续文件不再重复。